### 获取产品信息

In [ ]:
from app.infra.database import AsyncSessionFactory
from app.modules.product.service import ProductService

async with AsyncSessionFactory() as session:
    # 1.创建service对象
    service = ProductService(session)
    # 2.调用方法获取所有产品
    product_list = await service.find_by_category(None)

# 3.获取第一个产品
product = product_list[0]
print(product)

### 获取文件路径

In [ ]:
from app.core.config import APP_ROOT

clause_file_path =  f"{APP_ROOT}/data/raw/kb/{product.clause_name}"
print(clause_file_path)

### 加载文档(MinerU)

In [ ]:
from app.core.config import settings
from mineru import MinerU

# 1.MinerU客户端
client = MinerU(settings.rag.mineru_token)
# 2.执行精准解析
# file_content = client.extract(clause_file_path)
file_content = client.extract(r"G:\JYB\stage-2\面试题-智能体.pdf")

markdown = file_content.markdown
print(markdown)

### 优化文档

In [ ]:
from app.core.config import settings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model

# 1.准备系统提示词
MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()
# 2.准备模型
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=settings.llm.api_key,
    extra_body={"thinking":{"type":"disabled"}},
    max_tokens=10000    # 作用：防止大模型解析一半之后，不再解析其余的内容
)

# 3.调用模型执行优化
response:AIMessage = model.invoke([
    SystemMessage(content=MARKDOWN_OPTIMIZATION_PROMPT),
    HumanMessage(content=markdown)
])
markdown = response.content
print(markdown)

In [7]:
# 如果因为网络原因实在解析不出来文档，使用提前准备好的成品文档
with open(clause_file_path.replace('.pdf', '.md').replace("kb","final"), "r", encoding="utf-8") as f:
    markdown = f.read()

In [ ]:
from uuid import uuid4
from app.rag.models import ParentChunk
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# 准备文本切分器
# 1.1 MD文本切分器 - 切父块
md_splitter = MarkdownHeaderTextSplitter(
    # 切分依据
    headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3")
    ],
    # 不在正文中包含标题, 所有标题都存储在元数据中
    strip_headers=True
)

# 1.2 递归文本切分器
splitter = RecursiveCharacterTextSplitter(
    # 递归切分的依据
    separators=["\n\n", "\n", "。", ",", ""],

    # 每一个子块的目标大小
    chunk_size=400,
    # 滑动窗口的大小
    chunk_overlap=50
)

# 2. 将Markdown文本切分成父块
parent_chunks: list[Document] = md_splitter.split_text(str(markdown))
print(f"切分的父块长度：{len(parent_chunks)}")

# 3. 循环遍历所有的父块, 目标: 得到父块列表和子块列表
# 父块要写入到postgreSQL数据库, 所以要准备Document对象列表
parent_chunk_list: list[ParentChunk] = []
# 子块要写入到Milvus数据库, 所以要准备Documen对象列表
children_chunk_list: list[Document] = []

for parent_chunk in parent_chunks:
    # parent_chunk数据类型: Document父块

    # 3.0 准备标题列表
    section_path = list(parent_chunk.metadata.values())

    # 3.1 组装ParentChunk对象
    parent_chunk_obj = ParentChunk(
        id = uuid4(),
        product_id=product.id,
        clause_name=product.clause_name,
        # 存储当前父块的标题
        section_path=section_path,
        content=parent_chunk.page_content
    )
    parent_chunk_list.append(parent_chunk_obj)

    # 3.2 从父块切分子块 -
    children_chunks:list[Document] = splitter.split_documents([parent_chunk])

    # 准备多级标题字符串
    header_row = ""
    for section in section_path:
        header_row += section + "\n"

    for children_chunk in children_chunks:
        # children_chunk 数据类型: Document

        # 3.2.1 优化元数据
        children_chunk.metadata = {
            "parent_id": str(parent_chunk_obj.id),
            "product_id": product.id
        }

        # 3.2.2 优化子块内容 - 当前子块里面没有标题, 我们为子块添加标题行
        children_chunk.page_content = header_row + children_chunk.page_content

    # 在 for 循环外面将所有的当前父块的子块, 添加到列表中
    children_chunk_list.extend(children_chunks)

print(f"父块(ParentChunk)的数量: {len(parent_chunk_list)}")
print(f"子块(Document)的数量: {len(children_chunk_list)}")

In [ ]:
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings

embeddings = DashScopeEmbeddings(
        model="qwen3.7-text-embedding"
    )

# 2.准备VectorStore - 自动创建集合
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name="insurance_collection",
    connection_args={"uri":settings.rag.milvus_url},
    auto_id=True,
    vector_field=["dense", "sparse"],
    builtin_function=BM25BuiltInFunction(
        input_field_names="text",
        output_field_names="sparse",
        analyzer_params={"type":"chinese"},
        function_name="insurance_sparse_function"
    )
)

# 3.根据产品ID删除已有的所有子块 - 第一次执行会报错（不用管）
vector_store.delete(
    expr=f"product_id == {product.id}"
)

# 4.执行添加文档操作
vector_store.add_documents(children_chunk_list)

In [ ]:
from app.rag.repository import ParentChunkRepository

# 1.通过上下文管理器创建会话
async with AsyncSessionFactory() as session:
    # 2.开启事务
    async with session.begin():
        # 3.创建数据层对象
        repository = ParentChunkRepository(session)
        # 4.删除历史数据
        await repository.delete_by_product_id(product.id)
        # 5.添加新数据
        repository.add_all(parent_chunk_list)

In [3]:
from app.infra.database import AsyncSessionFactory
from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session:
    repository = ParentChunkRepository(session)
    result = await repository.list_by_ids(["9704db9f-7399-4927-a3fc-3b26d8433bcc"])
    print(result)

# from app.rag.repository import ParentChunkRepository
# from app.infra.database import AsyncSessionFactory
#
# async with AsyncSessionFactory() as session:
#     repository = ParentChunkRepository(session)
#     response = await repository.list_by_ids(["9704db9f-7399-4927-a3fc-3b26d8433bcc"])
#
#     print(response)

2026-10-26 12:14:20,546 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-26 12:14:20,548 INFO sqlalchemy.engine.Engine SELECT parent_chunks.id, parent_chunks.product_id, parent_chunks.section_path, parent_chunks.content, parent_chunks.clause_name, parent_chunks.created_at 
FROM parent_chunks 
WHERE parent_chunks.id IN ($1::UUID)
2026-10-26 12:14:20,549 INFO sqlalchemy.engine.Engine [cached since 51.02s ago] ('9704db9f-7399-4927-a3fc-3b26d8433bcc',)
[ParentChunk({'product_id': 1, 'section_path': ['信美相互互联网星辰守护B 款少儿重大疾病保险条款', '4. 合同效力的中止及恢复', '4.2 效力恢复'], 'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'content': '您可以申请恢复合同效力，经您与我们协商并达成协议，自您补交保险费及利息之日起，合同效力恢复。利息按照我们公布的保单贷款利率按复利（见9.29）计算。  \n自本合同效力中止之日起满2年您与我们未达成协议的，我们有权解除合同。我们解除本合同的，向您退还中止之日本合同的现金价值。', 'id': UUID('9704db9f-7399-4927-a3fc-3b26d8433bcc'), 'created_at': '2026-10-26 03:22:50'})]
2026-10-26 12:14:20,555 INFO sqlalchemy.engine.Engine ROLLBACK
